<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/5%20-%20Exemplos%20de%20Aplica%C3%A7%C3%A3o/5.4_mnist_rede_densa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 5.4 — MNIST com rede densa

Voltamos ao reconhecimento de dígitos manuscritos (MNIST, notebooks 3.2, 3.4 e 4.6). A entrada é uma imagem 28×28 em tons de cinza; a saída são 10 logits, um por dígito. Como no 4.6, a rede é **densa** (cada imagem é achatada num vetor de 784 pixels), agora com o pipeline dos notebooks 5.1–5.3 e uma rede bem maior. Acrescentamos aumento de dados (data augmentation): rotações aleatórias das imagens de treino.

## Objetivos

- Aplicar o pipeline do módulo 5 (rede densa com dropout, Adam, `StepLR`, parada antecipada, checkpoint) ao MNIST.
- Usar aumento de dados só no treino e entender por que o teste não deve ser alterado.
- Contar os parâmetros de uma rede densa e ver como o tamanho das camadas afeta custo computacional e acurácia.

## Importação das bibliotecas

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(1111)   # reprodutibilidade
np.random.seed(1111)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print("device:", device)

## Dados

As transformações de cada imagem:

- `transforms.ToTensor()`: converte a imagem em tensor `(1, 28, 28)` com valores em $[0, 1]$;
- `transforms.Normalize((0.1307,), (0.3081,))`: subtrai 0,1307 e divide por 0,3081, a média e o desvio-padrão dos pixels do treino do MNIST;
- `transforms.RandomRotation(15)`: gira a imagem por um ângulo sorteado entre −15° e 15°, **só no treino**. A cada época, a rede vê versões um pouco diferentes de cada dígito, o que ajuda a generalizar para letras inclinadas. O teste fica sem rotação: queremos medir o desempenho nas imagens como elas são, e sempre do mesmo jeito.

In [ ]:
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(15),
    transforms.Normalize((0.1307,), (0.3081,))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
dataset1 = datasets.MNIST(root="./data", train=True, download=True, transform=transform_train)
dataset2 = datasets.MNIST(root="./data", train=False, download=True, transform=transform_test)
print("treino:", len(dataset1), "| teste:", len(dataset2))

train_kwargs = {"batch_size": 1000, "shuffle": True}
test_kwargs = {"batch_size": 1000, "shuffle": False}
if use_cuda:
    cuda_kwargs = {"num_workers": 1, "pin_memory": True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

train_loader = DataLoader(dataset1, **train_kwargs)
test_loader = DataLoader(dataset2, **test_kwargs)
print("mini-batches por época:", len(train_loader))

In [ ]:
# A mesma imagem de treino, sorteada 6 vezes: cada leitura recebe uma rotação diferente
plt.figure(figsize=(9, 1.8))
for i in range(6):
    img, label = dataset1[0]
    plt.subplot(1, 6, i + 1)
    plt.imshow(img[0], cmap="gray")
    plt.axis("off")
plt.suptitle(f"Aumento de dados: dígito {label} com rotações aleatórias")
plt.show()

## Modelo

`x.view(-1, 784)` achata cada imagem `(1, 28, 28)` num vetor de 784 pixels; o `-1` deixa o PyTorch calcular o tamanho do mini-batch. Depois vêm três camadas escondidas (2048, 9216 e 128 neurônios, ReLU), dropout de 0,3 nas duas primeiras e 10 logits na saída.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(784, 2048)
        self.drop1 = nn.Dropout(0.3)
        self.hidden = nn.Linear(2048, 9216)
        self.drop2 = nn.Dropout(0.3)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)      # (N, 1, 28, 28) -> (N, 784)
        x = self.input(x)
        x = F.relu(x)
        x = self.drop1(x)
        x = self.hidden(x)
        x = F.relu(x)
        x = self.drop2(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)          # logits, (N, 10)
        return x

model = Net()
print(model)

## Treinamento

Funções `train` e `test` e laço com `StepLR`, parada antecipada e checkpoint, como no notebook 5.1. Esta rede é grande (veja o exercício 1): em CPU, cada época leva cerca de um minuto, e o notebook inteiro (duas redes, 10 épocas cada) leva mais de 20 minutos. No Colab, use uma GPU: menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU*.

In [ ]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()                       # liga o dropout
    running_loss = 0.0
    correct = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)            # logits, shape (N, n_classes)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)    # loss é a média do mini-batch; multiplicamos pelo tamanho para somar
        pred = output.argmax(dim=1)                   # classe com o maior logit
        correct += (pred == target).sum().item()
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = 100 * correct / len(train_loader.dataset)
    return train_loss, train_acc

In [ ]:
def test(model, device, test_loader, criterion):
    model.eval()                        # desliga o dropout
    test_loss = 0.0
    correct = 0
    with torch.no_grad():               # avaliação: sem gradientes
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
    test_loss /= len(test_loader.dataset)
    test_acc = 100 * correct / len(test_loader.dataset)
    return test_loss, test_acc

In [ ]:
n_epochs = 10   # em CPU, ~1 min por época; com GPU, poucos segundos
lr = 0.001
lr_decay = 5     # a cada lr_decay épocas, o StepLR multiplica a taxa de aprendizado por gamma=0.5
patience = 5      # parada antecipada: para se a acurácia de teste não melhorar por `patience` épocas
patience_counter = 0
best_acc = 0
ckpt_path = "melhor_modelo_mnist_denso.pt"

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

torch.manual_seed(1111)
model = Net().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()   # recebe logits (aplica log-softmax internamente)
scheduler = StepLR(optimizer, step_size=lr_decay, gamma=0.5)

print("Iniciando treinamento ...")
for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = train(model, device, train_loader, optimizer, criterion)
    test_loss, test_acc = test(model, device, test_loader, criterion)
    scheduler.step()

    print(f"Época {epoch:3d} | treino: custo {train_loss:.4f}, acc {train_acc:.2f}% | teste: custo {test_loss:.4f}, acc {test_acc:.2f}%")
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), ckpt_path)   # state_dict: dicionário com os tensores de pesos e bias
        print("    melhor época até agora: pesos salvos")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter == patience:
            print(f"Parada antecipada na época {epoch}")
            break

In [ ]:
# Recarrega os pesos da melhor época (os da última época podem ser piores)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_loss, test_acc = test(model, device, test_loader, criterion)
print(f"Melhor modelo | teste: custo {test_loss:.4f}, acurácia {test_acc:.2f}%")

## Avaliação

In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="treino")
plt.plot(range(1, len(history["test_loss"]) + 1), history["test_loss"], label="teste")
plt.title("Função de custo (cross-entropy)")
plt.xlabel("época")
plt.ylabel("custo")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history["train_acc"]) + 1), history["train_acc"], label="treino")
plt.plot(range(1, len(history["test_acc"]) + 1), history["test_acc"], label="teste")
plt.title("Acurácia")
plt.xlabel("época")
plt.ylabel("acurácia (%)")
plt.legend()
plt.grid(True)

plt.show()

Nas primeiras épocas, a acurácia de treino fica bem abaixo da de teste (86% contra 96% na primeira). Não é erro: a de treino é medida com o dropout ligado, sobre imagens giradas e como média ao longo da época, enquanto os pesos ainda estão mudando; a de teste usa o modelo do fim da época, sem dropout e sem rotação. Com o treino, as duas se aproximam.

## Exercícios

**1.** Conte os parâmetros (pesos e bias) de cada camada da rede e o total. Use `model.named_parameters()` e `p.numel()`. Que camada concentra a maior parte dos parâmetros? Compare o total com os 7.850 parâmetros da regressão softmax do notebook 3.2 (`nn.Linear(784, 10)`).

In [ ]:
# TODO: escreva sua solução aqui
print(f"{'total':31s} {total:>12,}")

*Escreva sua resposta aqui.*

**2.** Mostre 10 imagens do conjunto de teste que o melhor modelo classifica errado, com o rótulo correto e o previsto no título de cada uma. Os erros são compreensíveis?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

**3.** Troque a arquitetura por uma rede menor, `NetPequena`, com camadas escondidas de 512 e 256 neurônios (ReLU e dropout de 0,3 em cada uma) e 10 logits na saída. Treine com o mesmo número de épocas, otimizador e taxa de aprendizado, e compare o número de parâmetros e a acurácia de teste com os da rede original.

In [ ]:
class NetPequena(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: escreva sua solução aqui

    def forward(self, x):
        x = x.view(-1, 784)
        # TODO: escreva sua solução aqui
        return x

In [ ]:
torch.manual_seed(1111)
model_pequena = NetPequena().to(device)
optimizer_p = optim.Adam(model_pequena.parameters(), lr=lr)
scheduler_p = StepLR(optimizer_p, step_size=lr_decay, gamma=0.5)

inicio = time.time()
for epoch in range(1, n_epochs + 1):
    tr_loss, tr_acc = train(model_pequena, device, train_loader, optimizer_p, criterion)
    te_loss, te_acc = test(model_pequena, device, test_loader, criterion)
    scheduler_p.step()
    print(f"Época {epoch} | treino acc {tr_acc:.2f}% | teste acc {te_acc:.2f}%")

print(f"\nNetPequena: {sum(p.numel() for p in model_pequena.parameters()):,} parâmetros, "
      f"{time.time() - inicio:.0f} s de treino, acurácia de teste {te_acc:.2f}%")
print(f"Net:        {sum(p.numel() for p in model.parameters()):,} parâmetros, acurácia de teste (melhor época) {test_acc:.2f}%")

*Escreva sua resposta aqui.*

## Aprendizados

- O mesmo pipeline do módulo 5 serve para imagens: basta achatar cada imagem com `x.view(-1, 784)`.
- Aumento de dados (rotações aleatórias) é aplicado só no treino; o teste é sempre medido nas imagens originais.
- Com dropout e aumento de dados, a acurácia de treino pode ficar abaixo da de teste sem que haja erro.
- Numa rede densa, o número de parâmetros cresce com o produto dos tamanhos de camadas vizinhas; uma rede muito maior pode ter quase a mesma acurácia que uma pequena.